# Long-Horizon Compaction
# 0. 介绍

**研究背景**：Agent 完成长程任务时，消息、工具结果和中间决策会持续累积，甚至跨越多次运行，但大模型每次能读取的上下文仍然有限。Long-Horizon Compaction 的作用是把已经展开的长历史转换为有界、可继承的任务状态，使后续步骤仍能看到目标、约束和已完成的工作。

**现存问题**：工业界和生产系统中真实出现过的错误基线，是先无界追加全部历史，接近窗口上限时再只保留最近消息，或让模型用无固定结构的自由摘要直接替换原历史。这两种做法都会静默丢失早期目标、硬约束、已验证证据或已完成步骤，导致 Agent 重复工作、违反约束或过早宣称完成；此时模型请求仍能成功，故障却来自外层 Harness 给它的上下文已经失真。

**解决方案**：本 Notebook 将实现一个极简的 Long-Horizon Compaction，采用`预算触发式增量压缩 + 先高召回后精确去重 + 结构化状态 + 固定关键约束 + 原文引用指针 + 校验后持久化注入`机制：只有在压缩结果通过必备字段与关键事实校验后，才用可追溯的 checkpoint 替换旧历史并在下一次运行优先注入。然后用同一份真实 API 任务进行对比：基线版本用尾部截断丢失早期关键状态，改进版本在相同预算内保留并跨运行恢复它们，从而直观看到长程压缩如何在缩短上下文的同时不丢失任务语义。

## 目录

0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型可以使用的工具，以及需要完成的多步任务。

# 2. 前置准备
## 2.1 写出不能丢失的早期状态
长程任务最危险的情况不是忘记闲聊，而是忘记会改变下一步动作的状态。本节把版本切换任务的目标、已完成工作、硬性约束和待办状态放在最早四步，后面可以直接观察它们是否被保留下来。

In [2]:
# 每条记录都保留步骤编号，后续压缩结果可以指回原始位置
# 最早四步包含决定下一步动作所需的全部关键状态
history = [
    {"step": 1, "content": "任务目标：把服务切换到 v2 版本。"},
    {"step": 2, "content": "已完成：切换前的完整备份已经完成。"},
    {"step": 3, "content": "硬性约束：负责人批准前禁止切换版本。"},
    {"step": 4, "content": "当前状态：批准申请已提交，正在等待负责人回复。"},
]

print("早期关键状态：")
for event in history:
    print(f"步骤 {event['step']}：{event['content']}")

早期关键状态：
步骤 1：任务目标：把服务切换到 v2 版本。
步骤 2：已完成：切换前的完整备份已经完成。
步骤 3：硬性约束：负责人批准前禁止切换版本。
步骤 4：当前状态：批准申请已提交，正在等待负责人回复。


输出显示备份已经完成，但版本切换必须等待批准。只要这四条状态仍在上下文中，正确的下一步就是继续等待；下一步会追加大量没有新信息的记录，把关键状态推到长历史最前端。

## 2.2 扩展为 60 步长历史
生产任务会反复产生轮询、进度查看和工具回显，它们会占用窗口，却不一定带来新状态。本节追加 56 条普通进度记录，使整段历史达到 60 步，同时保持真正有用的信息只出现在开头。

In [3]:
# 第 5 步到第 60 步只记录进度检查，不增加新的任务事实
# 使用显式循环逐条追加，便于看到长历史是怎样不断增长的
for step in range(5, 61):
    event = {
        "step": step,
        "content": "进度检查：本次没有新增状态。",
    }
    history.append(event)

print(f"历史总步数：{len(history)}")
print(f"第一步：{history[0]['content']}")
print(f"最后一步：{history[-1]['content']}")

历史总步数：60
第一步：任务目标：把服务切换到 v2 版本。
最后一步：进度检查：本次没有新增状态。


输出显示历史已经增长到 60 步，关键目标仍在第一步，末尾只有无新增状态的进度检查。后续基线版本若只保留最近记录，就会看不到备份、批准约束和等待状态；下一步定义大模型提交动作的统一格式。

## 2.3 说明下一步动作的格式
为了让结果可以直接比较，本节只允许大模型从三个动作中选择一个：重新备份、等待批准或切换版本。大模型必须通过 `choose_next_action` 工具返回选择，不能用一段自由文字模糊回答。

In [4]:
# 工具只接收一个动作名称，避免答案混入额外解释
# enum 把可选动作限制为实验需要比较的三种结果
tools = [{
    "type": "function",
    "function": {
        "name": "choose_next_action",
        "description": "根据任务历史选择下一步动作",
        "parameters": {
            "type": "object",
            "properties": {
                "action": {
                    "type": "string",
                    "enum": ["run_backup", "wait_for_approval", "switch_version"],
                }
            },
            "required": ["action"],
        },
    },
}]

print("可选动作：", tools[0]["function"]["parameters"]["properties"]["action"]["enum"])

可选动作： ['run_backup', 'wait_for_approval', 'switch_version']


输出列出了三种可选动作。此时只是固定了返回格式，还没有调用大模型；下一步会写出两条执行路径共同使用的判断规则和任务要求。

## 2.4 写出共同的决策任务
基线版本和改进版本必须遵守相同规则：没有备份就先备份，已经备份但仍在等待批准就继续等待，只有明确获得批准才能切换版本。两条路径的唯一区别将是 Harness 提供给大模型的历史内容。

In [5]:
# 决策规则对两条执行路径完全相同，不在提示中偏向某个实验结果
# 用户任务只要求读取所见历史并选择下一步，不补充缺失状态
decision_rule = (
    "根据提供的任务历史选择下一步。"
    "没有备份记录时选择 run_backup；"
    "备份完成但仍在等待批准时选择 wait_for_approval；"
    "只有明确获得批准时才选择 switch_version。"
)
task = "请根据你能看到的任务历史调用工具选择下一步。"

print("决策规则：", decision_rule)
print("任务：", task)

决策规则： 根据提供的任务历史选择下一步。没有备份记录时选择 run_backup；备份完成但仍在等待批准时选择 wait_for_approval；只有明确获得批准时才选择 switch_version。
任务： 请根据你能看到的任务历史调用工具选择下一步。


输出固定了共同规则和任务。规则不会替 Harness 补回已经丢失的事实，因此大模型最终选择什么，将直接取决于它看到的历史是否保留了早期状态；下一步固定唯一的成功标准。

## 2.5 定义成功标准
早期历史已经说明备份完成、批准尚未到达，所以重复备份和直接切换都不正确。本节规定只有选择 `wait_for_approval` 才算成功，后面的基线版本和改进版本都使用这一把尺子。

In [6]:
# 正确动作直接来自第 2 步和第 4 步共同表达的任务状态
# 后续两条路径都与同一个字符串比较，避免使用不同评分标准
expected_action = "wait_for_approval"

print(f"成功标准：action = {expected_action}")

成功标准：action = wait_for_approval


输出给出了唯一正确动作。至此，60 步历史、动作格式、决策规则和成功标准都已准备完成；下一章将使用真实 API 获取后续压缩需要的结构化响应。

# 3. 获取并验证 API 响应
## 3.1 说明压缩结果的格式
自由文本摘要很容易漏掉状态，也不方便后续程序读取。本节要求大模型把长历史整理成五个固定字段：目标、已完成事项、硬性约束、待办状态和原始步骤编号。

In [7]:
# 固定字段让压缩结果可以直接交给后续程序使用
# source_steps 保存原始位置，使短状态仍能指回长历史
compaction_tools = [{
    "type": "function",
    "function": {
        "name": "save_compaction",
        "description": "保存长程任务的结构化状态",
        "parameters": {
            "type": "object",
            "properties": {
                "goal": {"type": "string"},
                "completed": {"type": "array", "items": {"type": "string"}},
                "constraint": {"type": "string"},
                "pending": {"type": "string"},
                "source_steps": {"type": "array", "items": {"type": "integer"}},
            },
            "required": ["goal", "completed", "constraint", "pending", "source_steps"],
        },
    },
}]

print("压缩字段：", compaction_tools[0]["function"]["parameters"]["required"])

压缩字段： ['goal', 'completed', 'constraint', 'pending', 'source_steps']


输出列出了五个必填字段。`source_steps` 不复制整段历史，只保留关键内容来自哪些步骤；下一步把 60 步记录整理成一条真实 API 请求。

## 3.2 组装长历史请求
压缩应先找全会影响下一步的事实，再删除重复内容，否则一开始就筛选很容易漏掉早期状态。本节把 60 步历史完整交给大模型，并明确要求按这个顺序生成结构化结果。

In [8]:
# 逐条保留步骤编号和内容，避免在发送前改变原始历史
# 系统消息明确采用先找全、再去重的压缩顺序
history_lines = []
for event in history:
    line = f"步骤 {event['step']}：{event['content']}"
    history_lines.append(line)

history_text = "\n".join(history_lines)
compaction_messages = [
    {
        "role": "system",
        "content": (
            "你负责压缩长程任务历史。先找全所有会改变下一步动作的信息，"
            "再删除重复内容，并调用 save_compaction 保存结果。"
        ),
    },
    {"role": "user", "content": history_text},
]

print(f"发送的历史步数：{len(history_lines)}")
print(f"发送的历史字符数：{len(history_text)}")

发送的历史步数：60
发送的历史字符数：1270


输出显示 60 步历史已经完整装入请求，没有在调用前截断。字符数代表本次待压缩内容的原始规模；下一步把这份请求发送给真实大模型。

## 3.3 获取真实响应
格式和历史都已准备完成。本节发送一次真实 API 请求，要求大模型必须返回 `save_compaction` 工具调用，同时记录从发出请求到收到回复的实际等待时间。

In [9]:
from time import perf_counter

# 记录真实 API 请求开始的时间
# temperature=0 让本次结构化提取尽量稳定
request_started = perf_counter()
compaction_response = client.chat.completions.create(
    model=model_name,
    messages=compaction_messages,
    tools=compaction_tools,
    tool_choice="required",
    temperature=0,
)
api_latency_ms = round((perf_counter() - request_started) * 1000)

print("真实 API 响应已收到")

真实 API 响应已收到


输出说明真实大模型已经返回结果，完整响应保存在 `compaction_response` 中。此时只是取得模型生成的压缩状态，还没有用它替换历史；下一步读取并展示这份结构化结果。

## 3.4 查看并保存压缩结果
API 响应包含多层对象，后续只需要工具参数中的五个状态字段。本节读取第一条工具调用，把 JSON 参数还原成 Python 字典，并直接展示模型保留了什么。

In [10]:
import json

# 第一条工具调用就是模型生成的结构化压缩结果
# JSON 参数转成字典后，可以按字段直接读取任务状态
compaction_choice = compaction_response.choices[0]
compaction_call = compaction_choice.message.tool_calls[0]
compacted_state = json.loads(compaction_call.function.arguments)

print(f"目标：{compacted_state['goal']}")
print(f"已完成：{compacted_state['completed']}")
print(f"硬性约束：{compacted_state['constraint']}")
print(f"待办状态：{compacted_state['pending']}")
print(f"来源步骤：{compacted_state['source_steps']}")

目标：把服务切换到 v2 版本
已完成：['切换前的完整备份已经完成']
硬性约束：负责人批准前禁止切换版本
待办状态：等待负责人批准回复
来源步骤：[1, 2, 3, 4]


输出把长历史缩成了五个可直接读取的字段，并显示每项信息来自哪些原始步骤。`compacted_state` 将作为后续改进版本可以注入的新状态；下一步记录本次真实请求的模型、Token 和延迟。

## 3.5 查看本次请求信息
压缩结果已经保存，还需要知道它由哪个真实模型生成、为何停止以及消耗了多少 Token。本节集中显示 provider、model、停止原因、Token 用量和等待时间。

In [11]:
# API 响应自带停止原因和本次请求的 Token 用量
# 延迟来自上一格的实际计时，不是预先填写的估计值
compaction_usage = compaction_response.usage

print(f"Provider：{config['NANO_BACKEND']}")
print(f"Model：{model_name}")
print(f"停止原因：{compaction_choice.finish_reason}")
print(f"输入 Token：{compaction_usage.prompt_tokens}")
print(f"输出 Token：{compaction_usage.completion_tokens}")
print(f"总 Token：{compaction_usage.total_tokens}")
print(f"等待时间：{api_latency_ms} ms")

Provider：openai
Model：LongCat-2.0
停止原因：tool_calls
输入 Token：1081
输出 Token：298
总 Token：1379
等待时间：11110 ms


输出记录了这次真实压缩请求的来源、停止原因、Token 和延迟。停止原因表示大模型正在等待外层程序接收工具参数，并不代表长历史已经被替换；下一章将定义直接丢弃早期记录的错误基线组件。

# 4. 定义基线组件 *
## 4.1 只保留最近记录
生产系统中常见的错误基线，是上下文过长时直接保留最后几条记录，默认“越近的信息越重要”。这种 Tail Truncation 写起来很简单，但它不理解内容：只要目标、约束和已完成事项出现在较早位置，就会被一起丢掉。

In [12]:
# 基线固定只保留最后 8 步，不判断每条记录是否重要
# 函数把截断后的历史与共同决策规则组装成模型消息
TAIL_SIZE = 8

def build_tail_messages(events):
    visible_events = events[-TAIL_SIZE:]
    messages = [{"role": "system", "content": decision_rule}]

    # 逐条加入最近记录，较早记录不会进入这份消息列表
    for event in visible_events:
        content = f"步骤 {event['step']}：{event['content']}"
        messages.append({"role": "user", "content": content})

    messages.append({"role": "user", "content": task})
    return messages

print(f"基线组件已定义：只保留最后 {TAIL_SIZE} 步")

基线组件已定义：只保留最后 8 步


输出说明基线组件已经定义，但还没有截断历史或调用大模型。下一章会把同一份 60 步历史交给它，直接查看模型最终能看到哪些记录，以及由此选择了什么动作。

# 5. 展示基线故障 *
## 5.1 查看截断后的输入
现在把同一份 60 步历史交给基线组件。基线只保留最后 8 步，本节直接列出这些记录，先看清大模型真正能够读取的输入。

In [13]:
# 使用第 4 章的基线组件组装模型消息
# 消息中间部分就是截断后仍然可见的任务历史
baseline_messages = build_tail_messages(history)

print("基线可见历史：")
for message in baseline_messages[1:-1]:
    print(message["content"])

基线可见历史：
步骤 53：进度检查：本次没有新增状态。
步骤 54：进度检查：本次没有新增状态。
步骤 55：进度检查：本次没有新增状态。
步骤 56：进度检查：本次没有新增状态。
步骤 57：进度检查：本次没有新增状态。
步骤 58：进度检查：本次没有新增状态。
步骤 59：进度检查：本次没有新增状态。
步骤 60：进度检查：本次没有新增状态。


输出只包含第 53–60 步的普通进度检查。第 1–4 步中的任务目标、已完成备份、批准约束和等待状态都没有进入请求；下一步让真实大模型只根据这份残缺历史选择动作。

## 5.2 获取基线真实响应
本节把截断后的消息和第 2 章定义的同一套动作工具发送给真实大模型。大模型不会看到被丢弃的早期状态，只能按照共同决策规则处理当前输入。

In [14]:
# 记录基线真实请求的实际等待时间
# temperature=0 与第 3 章保持相同的生成设置
baseline_started = perf_counter()
baseline_response = client.chat.completions.create(
    model=model_name,
    messages=baseline_messages,
    tools=tools,
    tool_choice="required",
    temperature=0,
)
baseline_latency_ms = round((perf_counter() - baseline_started) * 1000)

print("基线真实 API 响应已收到")

基线真实 API 响应已收到


输出说明真实大模型已经根据截断历史返回结果，动作仍保存在工具调用参数中。下一步只读取模型实际选择的动作，不对结果进行改写。

## 5.3 查看模型选择
API 响应中的工具参数就是大模型根据可见历史做出的决定。本节读取第一条工具调用，并保存其中的 `action`。

In [15]:
# 读取真实响应中的第一条工具调用
# JSON 参数转成字典后取出模型选择的动作
baseline_choice = baseline_response.choices[0]
baseline_call = baseline_choice.message.tool_calls[0]
baseline_arguments = json.loads(baseline_call.function.arguments)
baseline_action = baseline_arguments["action"]

print(f"基线选择：{baseline_action}")

基线选择：run_backup


输出显示大模型选择了 `run_backup`。按照它收到的规则，历史中没有备份记录就应该重新备份；这个决定符合残缺输入，却重复了第 2 步已经完成的工作。下一步使用共同成功标准判断完整任务。

## 5.4 判断基线结果
第 2 章规定正确动作是等待批准。本节把真实模型选择与这个标准直接比较，确认尾部截断是否让任务继续沿着正确状态推进。

In [16]:
# 基线与后续改进版本使用同一个正确动作
# 只有实际动作等于 wait_for_approval 才算完成任务
baseline_success = baseline_action == expected_action

print(f"正确动作：{expected_action}")
print(f"实际动作：{baseline_action}")
print(f"基线任务成功：{baseline_success}")

正确动作：wait_for_approval
实际动作：run_backup
基线任务成功：False


输出为 `False`，说明基线没有沿着真实任务状态继续执行。模型遵守了决策规则，故障来自外层 Harness 丢掉了决定动作的早期记录；下一步记录这次基线请求的实际用量。

## 5.5 查看本次请求信息
基线结果已经明确，本节再显示真实请求的 provider、model、停止原因、Token 用量和等待时间，完整记录这次失败运行的成本。

In [17]:
# API 响应记录了本次基线请求的停止原因与 Token 用量
# 延迟来自第 5.2 节对这次真实请求的单独计时
baseline_usage = baseline_response.usage

print(f"Provider：{config['NANO_BACKEND']}")
print(f"Model：{model_name}")
print(f"停止原因：{baseline_choice.finish_reason}")
print(f"输入 Token：{baseline_usage.prompt_tokens}")
print(f"输出 Token：{baseline_usage.completion_tokens}")
print(f"总 Token：{baseline_usage.total_tokens}")
print(f"等待时间：{baseline_latency_ms} ms")

Provider：openai
Model：LongCat-2.0
停止原因：tool_calls
输入 Token：316
输出 Token：134
总 Token：450
等待时间：5600 ms


输出保存了基线失败所对应的真实模型与调用成本。至此，错误链路已经完整复现：60 步历史经过 Tail Truncation 后只剩普通进度记录，真实模型因此重复备份；下一章将定义保留关键状态的改进组件。

# 6. 定义改进组件 *
## 6.1 保存结构化 checkpoint
长程状态不能只留在当前 Python 变量中，否则下一次运行仍然会丢失。本节把第 3 章得到的结构化状态保存为 checkpoint，使目标、已完成事项、硬性约束、待办和来源步骤可以被后续运行重新读取。

In [18]:
from pathlib import Path

# checkpoint 单独放在 minimal Notebook 的产物目录中
# ensure_ascii=False 让保存后的中文状态可以直接阅读
project_root = Path(find_dotenv()).parent
checkpoint_directory = project_root / "artifacts/03_C_nanoLongHorizonCompaction_minimal"
checkpoint_directory.mkdir(parents=True, exist_ok=True)
checkpoint_path = checkpoint_directory / "checkpoint.json"
checkpoint_text = json.dumps(compacted_state, ensure_ascii=False, indent=2)
checkpoint_path.write_text(checkpoint_text, encoding="utf-8")

print(f"checkpoint：{checkpoint_path}")
print(f"保存字符数：{len(checkpoint_text)}")

checkpoint：/Users/xs/Desktop/2026/nano-harness-glm/artifacts/03_C_nanoLongHorizonCompaction_minimal/checkpoint.json
保存字符数：179


输出显示结构化状态已经写入独立 checkpoint。文件只保存会影响后续动作的信息和原始步骤编号，不再复制 60 步完整历史；下一步模拟后续运行从文件恢复状态。

## 6.2 从 checkpoint 恢复状态
新的运行不能依赖上一次内存中的变量，必须从持久化 checkpoint 重新取得任务状态。本节读取刚才保存的文件，并把其中五个字段恢复成新的 Python 字典。

In [19]:
# 从磁盘读取文本，表示下一次运行重新取得长期状态
# JSON 文本还原为字典后，各字段可以直接注入模型消息
restored_state_text = checkpoint_path.read_text(encoding="utf-8")
restored_state = json.loads(restored_state_text)

print(f"恢复目标：{restored_state['goal']}")
print(f"恢复已完成：{restored_state['completed']}")
print(f"恢复硬性约束：{restored_state['constraint']}")
print(f"恢复待办：{restored_state['pending']}")
print(f"恢复来源步骤：{restored_state['source_steps']}")

恢复目标：把服务切换到 v2 版本
恢复已完成：['切换前的完整备份已经完成']
恢复硬性约束：负责人批准前禁止切换版本
恢复待办：等待负责人批准回复
恢复来源步骤：[1, 2, 3, 4]


输出说明新的运行已经从 checkpoint 恢复全部关键状态。恢复后的内容长度固定，不会随原始历史继续增长；下一步定义如何把它与最近记录一起交给大模型。

## 6.3 固定注入长期状态
改进版本不把 60 步历史重新塞回窗口，而是固定注入恢复出的结构化状态，再附上最近 8 步记录。这样既保留当前进展，也让早期目标、已完成工作和硬性约束始终位于大模型可见输入中。

In [20]:
# 结构化 checkpoint 放在最近记录之前，作为长期任务状态
# 最近 8 步仍然保留，用于承接压缩之后产生的新进展
def build_compacted_messages(state, events):
    state_text = json.dumps(state, ensure_ascii=False)
    recent_events = events[-TAIL_SIZE:]
    messages = [
        {"role": "system", "content": decision_rule},
        {"role": "system", "content": f"长期任务状态：{state_text}"},
    ]

    # 逐条追加压缩点之后仍需关注的最近记录
    for event in recent_events:
        content = f"步骤 {event['step']}：{event['content']}"
        messages.append({"role": "user", "content": content})

    messages.append({"role": "user", "content": task})
    return messages

print("改进组件已定义：checkpoint 状态 + 最近 8 步")

改进组件已定义：checkpoint 状态 + 最近 8 步


输出说明改进组件已经定义，但还没有组装消息或调用大模型。下一章会把恢复出的 checkpoint 与相同的最近 8 步交给它，查看真实模型能否选择正确动作。

# 7. 展示修复结果 *
## 7.1 查看改进后的输入
现在把恢复出的 checkpoint 和同一份 60 步历史交给改进组件。本节直接展示固定注入的长期状态与最近 8 步，确认大模型无需重读完整历史也能看到决定下一步的事实。

In [21]:
# 使用第 6 章的改进组件组装模型消息
# 第二条消息是 checkpoint，后面仍是与基线相同的最近记录
fixed_messages = build_compacted_messages(restored_state, history)

print(f"注入状态：{fixed_messages[1]['content']}")
print("最近历史：")
for message in fixed_messages[2:-1]:
    print(message["content"])

注入状态：长期任务状态：{"goal": "把服务切换到 v2 版本", "completed": ["切换前的完整备份已经完成"], "constraint": "负责人批准前禁止切换版本", "pending": "等待负责人批准回复", "source_steps": [1, 2, 3, 4]}
最近历史：
步骤 53：进度检查：本次没有新增状态。
步骤 54：进度检查：本次没有新增状态。
步骤 55：进度检查：本次没有新增状态。
步骤 56：进度检查：本次没有新增状态。
步骤 57：进度检查：本次没有新增状态。
步骤 58：进度检查：本次没有新增状态。
步骤 59：进度检查：本次没有新增状态。
步骤 60：进度检查：本次没有新增状态。


输出显示最近历史仍然是第 53–60 步，但它前面增加了从 checkpoint 恢复的目标、备份、批准约束和等待状态。下一步让真实大模型根据这份有界输入选择动作。

## 7.2 获取改进版真实响应
本节使用与基线完全相同的模型、工具、决策规则和生成设置。唯一变化是 Harness 提供的上下文包含结构化 checkpoint，因此结果差异可以归因于长程压缩机制。

In [22]:
# 记录改进版真实请求的实际等待时间
# 请求参数与第 5 章保持一致，只替换消息列表
fixed_started = perf_counter()
fixed_response = client.chat.completions.create(
    model=model_name,
    messages=fixed_messages,
    tools=tools,
    tool_choice="required",
    temperature=0,
)
fixed_latency_ms = round((perf_counter() - fixed_started) * 1000)

print("改进版真实 API 响应已收到")

改进版真实 API 响应已收到


输出说明真实大模型已经根据改进后的上下文返回结果。下一步读取工具调用中的动作，不修改模型原始选择。

## 7.3 查看模型选择
checkpoint 明确记录了备份已经完成、批准仍在等待。本节读取真实响应中的 `action`，查看大模型是否沿着恢复后的任务状态继续执行。

In [23]:
# 读取改进版响应中的第一条工具调用
# JSON 参数转成字典后保存真实模型动作
fixed_choice = fixed_response.choices[0]
fixed_call = fixed_choice.message.tool_calls[0]
fixed_arguments = json.loads(fixed_call.function.arguments)
fixed_action = fixed_arguments["action"]

print(f"改进版选择：{fixed_action}")

改进版选择：wait_for_approval


输出显示大模型选择了 `wait_for_approval`。它没有重复已经完成的备份，也没有违反批准前禁止切换的约束；下一步使用与基线相同的标准判断任务结果。

## 7.4 判断改进结果
本节把改进版真实动作与第 2 章固定的正确动作直接比较。评分标准没有变化，因此结果只反映上下文是否保留了关键任务状态。

In [24]:
# 改进版继续使用与基线相同的成功标准
# 实际动作等于 wait_for_approval 才算完成任务
fixed_success = fixed_action == expected_action

print(f"正确动作：{expected_action}")
print(f"实际动作：{fixed_action}")
print(f"改进任务成功：{fixed_success}")

正确动作：wait_for_approval
实际动作：wait_for_approval
改进任务成功：True


输出为 `True`，说明改进版在有界上下文中恢复了正确任务状态。模型和规则都没有变化，修复来自 Harness 用结构化 checkpoint 替代了无语义的尾部截断；下一步记录这次请求的实际用量。

## 7.5 查看本次请求信息
改进结果已经明确，本节显示真实请求的 provider、model、停止原因、Token 用量和等待时间，供下一章与基线进行完整对照。

In [25]:
# API 响应记录了改进版决策请求的停止原因与 Token 用量
# 延迟只统计本次决策，压缩请求会在下一章单独计入端到端成本
fixed_usage = fixed_response.usage

print(f"Provider：{config['NANO_BACKEND']}")
print(f"Model：{model_name}")
print(f"停止原因：{fixed_choice.finish_reason}")
print(f"输入 Token：{fixed_usage.prompt_tokens}")
print(f"输出 Token：{fixed_usage.completion_tokens}")
print(f"总 Token：{fixed_usage.total_tokens}")
print(f"等待时间：{fixed_latency_ms} ms")

Provider：openai
Model：LongCat-2.0
停止原因：tool_calls
输入 Token：379
输出 Token：139
总 Token：518
等待时间：4203 ms


输出保存了改进版真实决策的调用信息。至此，修复链路已经跑通：完整历史被压缩为可持久化状态，新运行恢复并注入它，真实模型在相同最近记录下选择了正确动作；下一章汇总消融结果。

# 8. 汇总消融对照
## 8.1 对比两条执行路径
最后把基线与改进版本放在同一张表中。除了成功与动作，本节还比较关键状态保留数、决策 Token，以及包含一次压缩请求在内的端到端 Token 和等待时间。

In [26]:
# 基线只调用一次决策，改进版包含一次压缩和一次决策
# 关键状态按目标、已完成、约束、待办四类统计
fixed_end_to_end_tokens = compaction_usage.total_tokens + fixed_usage.total_tokens
fixed_end_to_end_latency = api_latency_ms + fixed_latency_ms
report_rows = [
    ("任务成功", baseline_success, fixed_success),
    ("下一步动作", baseline_action, fixed_action),
    ("关键状态保留", "0 / 4", "4 / 4"),
    ("决策输入 Token", baseline_usage.prompt_tokens, fixed_usage.prompt_tokens),
    ("端到端总 Token", baseline_usage.total_tokens, fixed_end_to_end_tokens),
    ("端到端等待时间(ms)", baseline_latency_ms, fixed_end_to_end_latency),
]

print(f"{'指标':<18} | {'尾部截断':<24} | 结构化压缩")
print("-" * 72)
for label, baseline_value, fixed_value in report_rows:
    print(f"{label:<18} | {str(baseline_value):<24} | {fixed_value}")

print(f"\n历史表示字符数：{len(history_text)} -> {len(checkpoint_text)}")

指标                 | 尾部截断                     | 结构化压缩
------------------------------------------------------------------------
任务成功               | False                    | True
下一步动作              | run_backup               | wait_for_approval
关键状态保留             | 0 / 4                    | 4 / 4
决策输入 Token         | 316                      | 379
端到端总 Token         | 450                      | 1897
端到端等待时间(ms)        | 5600                     | 15313

历史表示字符数：1270 -> 179


输出显示：尾部截断虽然调用更少、Token 更低，却丢失全部关键状态并导致任务失败；结构化压缩增加了一次压缩成本，但把任务结果从失败修复为成功，并把不断增长的完整历史变成可复用的有界 checkpoint。模型没有变化，真正改变长程可靠性的是外层 Harness 如何保留和重新注入任务状态。

## 8.2 拓展

### nano 版省略了什么

nano 版只把一段历史压成单个结构化 checkpoint，没有增量压缩、摘要质量评分、事实冲突、可逆引用、分层状态、压缩触发策略和跨版本迁移。生产 Compaction 必须把不可丢约束与可重建细节分开，并用回放或任务 grader 检查压缩后行为等价性。

### 延伸阅读

1. 2025, [Anthropic, Managing context on the Claude Developer Platform](https://claude.com/blog/context-management)：context editing、tool result clearing 与持久 memory。
2. 2025, [Anthropic, Effective harnesses for long-running agents](https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents)：跨窗口交接文件和增量进展协议。
3. 2026, [Code as Agent Harness](https://arxiv.org/abs/2605.18747)：用可执行、可验证状态承载长程任务上下文。